# Workshop 2 · Power BI Dataset Performance · Exercise

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

In this workshop you build the Day 2 serving layer for Power BI: a monthly aggregate, an incremental-refresh view and a KPI table. This is not a repeat of Day 1 modeling; it is the Power BI dataset performance layer on top of the Gold model.


## Business Scenario

RetailHub's executive dashboard needs fast KPI cards and trends, while analysts still need drill-through detail. Importing every line for every report page is wasteful, and DirectQuery over wide line-grain data can be expensive. Build serving objects that make both paths explicit.


## Success Criteria

A correct solution must satisfy:

- `gold.fact_sales_dashboard_monthly` exists at monthly x region x category x channel grain,
- `gold.v_fact_sales_incremental` exposes `order_datetime` for Power BI incremental refresh,
- `gold.kpi_monthly` contains one row per `year_month`,
- monthly totals reconcile to `gold.fact_sales_dashboard` for completed rows,
- incremental refresh window uses a half-open interval,
- the BI contract states source object, grain, model mode and refresh expectation.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before starting the workshop.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")


## Source Baseline

Before creating serving objects, inspect the line-grain dashboard table that Workshop 1 produced.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM gold.fact_sales_dashboard


## Task 1: Build `gold.fact_sales_dashboard_monthly`

Definition: this is an Import-friendly aggregate for summary pages. Grain must be `year_month x customer_region x category x channel`.

Required measures: `line_rows`, `completed_orders`, `quantity`, `revenue`, `gross_margin`, `margin_rate_pct`.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales_dashboard_monthly.
-- Source: gold.fact_sales_dashboard.
-- Filter: completed rows only.
-- Grain: year_month, customer_region, category, channel.
-- Required: COMMENT on the table.


In [ ]:
%sql
SELECT
  COUNT(*) AS aggregate_rows,
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS distinct_grain_keys,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.fact_sales_dashboard_monthly


## Task 2: Reconcile Monthly Aggregate to Detail

Definition: reconciliation proves that the aggregate did not drop or duplicate completed revenue.


In [ ]:
%sql
-- TODO: return detail_revenue, aggregate_revenue and revenue_diff.
-- detail source: gold.fact_sales_dashboard where is_completed.
-- aggregate source: gold.fact_sales_dashboard_monthly.


## Task 3: Build `gold.v_fact_sales_incremental`

Definition: this view keeps line-grain detail but adds `order_datetime` as a timestamp column for Power BI incremental refresh parameters.


In [ ]:
%sql
-- TODO: create or replace gold.v_fact_sales_incremental.
-- Include all columns from gold.fact_sales_dashboard.
-- Add order_datetime = CAST(order_date AS TIMESTAMP).


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_datetime) AS min_order_datetime,
  MAX(order_datetime) AS max_order_datetime
FROM gold.v_fact_sales_incremental


## Task 4: Test Incremental Refresh Window

Use a half-open interval: `order_datetime >= RangeStart AND order_datetime < RangeEnd`.


In [ ]:
%sql
-- TODO: simulate RangeStart = 2024-01-01 and RangeEnd = 2024-02-01.
-- Return rows, min/max order date and completed revenue for that window.


## Task 5: Build `gold.kpi_monthly`

Definition: this table supports KPI cards and refresh sanity checks. Grain must be one row per `year_month`.

Required measures: revenue, gross margin, completed orders, returned orders, margin rate %, return rate %, average order value.


In [ ]:
%sql
-- TODO: create or replace gold.kpi_monthly.
-- Source: gold.fact_sales_dashboard.
-- Grain: one row per year_month.
-- Required: protect divisions with NULLIF.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT year_month) AS distinct_months,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.kpi_monthly


## Task 6: BI Contract

Return a table that documents the serving objects, intended Power BI mode and refresh expectation.


In [ ]:
%sql
-- TODO: return columns: object_name, grain, recommended_mode, refresh_expectation, primary_use.


## Completion Checklist

Before moving to Day2_02, confirm:

- monthly aggregate has unique grain keys,
- aggregate revenue reconciles to detail completed revenue,
- incremental view has `order_datetime`,
- KPI table has one row per month,
- BI contract can be handed to the Power BI owner.
